In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%ls

drive/  sample_data/


In [3]:
seeker_train_path = 'datasets/GoRecDial/raw_GoRecDial/dialrec-Seeker-all-train.txt'
seeker_test_path = 'datasets/GoRecDial/raw_GoRecDial/dialrec-Seeker-all-test.txt'
expert_train_path = 'datasets/GoRecDial/raw_GoRecDial/dialrec-Expert-all-train.txt'
expert_test_path = 'datasets/GoRecDial/raw_GoRecDial/dialrec-Expert-all-test.txt'

MID_pattern = r'(MID\d+)'
recommend_ground_truth = r'recommend_ground_truth:(MID\d+)'
# movieName_pattern = r'(MID\d+):(.*?) is\s+a\s+'
# movieName_pattern = r'MID{mid}:(.*?) is'





import re
import json
from tqdm import tqdm
import pandas as pd

In [8]:
def get_mid_to_movieName_dict(path):
  mid_to_movieName = {}
  # count = 0
  with open(path, 'r') as file:
    for line in tqdm(file, desc="Building mid_to_name list"):
      # count += 1
      # if count == 2:
      #   break
      if 'turn:0' in line:
        movie_mid = re.findall(MID_pattern, line)

        # Method 2:
        for mid in movie_mid:
          mid_tag = f'{mid}:'
          tmp = line.split(mid_tag)

          # Discard Ground Truth recommend movie
          if len(tmp) <= 1:
              break
          tmp = tmp[1]
          tmp_name = tmp.split(' is ')[0]
          mid_to_movieName[mid] = tmp_name
          # print(f'{mid} {tmp_name}')

  # # Manual addition
  # mid_to_movieName['MID819'] = 'Stefano Quantestorie'
  # mid_to_movieName['MID3111'] = 'Places in the Heart (1984)'
  # mid_to_movieName['MID5128'] = 'Queen of the Damned'
  # mid_to_movieName['MID3597'] = 'Whipped'
  # mid_to_movieName['MID779'] = '\'Til There Was You'
  # mid_to_movieName['MID114265'] = 'Laggies'
  # mid_to_movieName['MID4628'] = 'New York Stories'
  # mid_to_movieName['MID2075'] = 'Mephisto'
  # mid_to_movieName['MID621'] = 'My Favorite Season ()'
  # mid_to_movieName['MID4342'] = 'Big Eden'

  return mid_to_movieName.copy()

seek_train_movieName_dict = get_mid_to_movieName_dict(seeker_train_path)
expert_train_movieName_dict = get_mid_to_movieName_dict(expert_train_path)
seek_train_movieName_dict.update(expert_train_movieName_dict)
seek_test_movieName_dict = get_mid_to_movieName_dict(seeker_test_path)
expert_test_movieName_dict = get_mid_to_movieName_dict(expert_test_path)
seek_test_movieName_dict.update(expert_test_movieName_dict)
print(f'\nseek_train has {len(seek_train_movieName_dict)} movies')
print(f'seek_test has {len(seek_test_movieName_dict)} movies')
print(f'expert_train has {len(expert_train_movieName_dict)} movies')
print(f'expert_test has {len(expert_test_movieName_dict)} movies')


Building mid_to_name list: 80844it [00:00, 96733.53it/s]
Building mid_to_name list: 73312it [00:00, 90834.19it/s]
Building mid_to_name list: 8800it [00:00, 90986.99it/s]
Building mid_to_name list: 7948it [00:00, 90760.00it/s]


seek_train has 3677 movies
seek_test has 1692 movies
expert_train has 3335 movies
expert_test has 1378 movies


In [40]:
import random

def get_train_dataset(path, name='GoRecDial_Train', batch_size=1000, use_mentioned_movie_only=False, shuffle_dialgoue=False):
    # Create output file names
    json_path = f"datasets/GoRecDial/GoRecDial_train.jsonl"
    csv_path = f"datasets/GoRecDial/GoRecDial_train.csv"

    # Clear any existing output files
    with open(json_path, 'w') as f:
        pass

    batch_data = []
    tmp = {'movies': [], 'full_situation': ''}
    if shuffle_dialgoue:
      tmp = {'movies': [], 'full_situation': []}
    mentioned_movies = set()
    counter = 0

    # Process in batches
    with open(path, 'r') as seeker_train_file:
        for seeker_line in tqdm(seeker_train_file, desc=f"Processing {name} File"):
            if not seeker_line.strip():
                continue

            if 'turn:0' in seeker_line:
                if len(tmp['movies']) != 0:
                    # Add completed conversation to batch
                    if use_mentioned_movie_only:
                      # print(f'mentioned_movies: {mentioned_movies}')
                      tmp['movies'] = list(mentioned_movies.copy())
                    if shuffle_dialgoue:
                      random.shuffle(tmp['full_situation'])
                      tmp_string = " ".join(tmp['full_situation'])
                      tmp['full_situation'] = tmp_string
                    batch_data.append(tmp.copy())
                    counter += 1

                    # Write batch to disk if it reaches batch size
                    if counter % batch_size == 0:
                        write_batch_to_jsonl(batch_data, json_path)
                        batch_data = []  # Clear batch from memory

                tmp = {'movies': [], 'full_situation': ''}
                if shuffle_dialgoue:
                  tmp = {'movies': [], 'full_situation': []}
                mentioned_movies.clear()
                movie_list = re.findall(MID_pattern, seeker_line)
                # print(seeker_line)
                # print(movie_list)
                tmp['movies'] = [seek_train_movieName_dict[mid] for mid in movie_list]


                try:
                    the_first_question = re.findall(r'labels:(.+?)(?=\s*reward:)', seeker_line)
                    if the_first_question and len(the_first_question) > 0:
                        if not any(char in the_first_question[0] for char in ['.', ',', '?', '!']):
                          if not shuffle_dialgoue:
                            tmp['full_situation'] += the_first_question[0] + '.'
                          else:
                            tmp['full_situation'].append(the_first_question[0] + '.')
                        else:
                          if not shuffle_dialgoue:
                            tmp['full_situation'] += the_first_question[0]
                          else:
                            tmp['full_situation'].append(the_first_question[0])
                except Exception as e:
                    print(f"Error processing first question: {e}")
                    print(seeker_line)

            # Process dialogue lines
            else:
                try:
                    text = re.findall(r'text:(.*?)(?=\tlabels:)', seeker_line)
                    response_for_text = re.findall(r'labels:(.+?)(?=\s*reward:)', seeker_line)

                    if text and response_for_text and len(text) > 0 and len(response_for_text) > 0:
                      # Replace any MID with real movie name
                      if 'MID' in text[0]:
                        for mid in re.findall(MID_pattern, text[0]):
                          mentioned_movies.add(seek_train_movieName_dict[mid])
                          text[0] = text[0].replace(mid, seek_train_movieName_dict[mid])

                      if 'MID' in response_for_text[0]:
                        for mid in re.findall(MID_pattern, response_for_text[0]):
                          mentioned_movies.add(seek_train_movieName_dict[mid])
                          response_for_text[0] = response_for_text[0].replace(mid, seek_train_movieName_dict[mid])

                      if not any(char in response_for_text[0] for char in ['.', ',', '?', '!']):
                        if not shuffle_dialgoue:
                          tmp['full_situation'] += text[0] + response_for_text[0] + '.'
                        else:
                          tmp['full_situation'].append(text[0])
                          tmp['full_situation'].append(response_for_text[0] + '.')
                      else:
                        if not shuffle_dialgoue:
                          tmp['full_situation'] += text[0] + response_for_text[0]
                        else:
                          tmp['full_situation'].append(text[0])
                          tmp['full_situation'].append(response_for_text[0])
                except Exception as e:
                    print(f"Error processing dialogue line: {e}")
                    print(f"Text matches: {text if 'text' in locals() else 'Not found'}")
                    print(f"Response matches: {response_for_text if 'response_for_text' in locals() else 'Not found'}")
                    print(seeker_line)
                    continue

    # Add the last conversation and write remaining batch
    if len(tmp['movies']) != 0:
        if use_mentioned_movie_only:
          # print(f'mentioned_movies: {mentioned_movies}')
          tmp['movies'] = list(mentioned_movies.copy())
        if shuffle_dialgoue:
          random.shuffle(tmp['full_situation'])
          tmp_string = " ".join(tmp['full_situation'])
          tmp['full_situation'] = tmp_string
        batch_data.append(tmp.copy())

    if batch_data:
        write_batch_to_jsonl(batch_data, json_path)

    # Convert the JSONL file to CSV
    jsonl_to_csv(json_path, csv_path)

    return csv_path

def write_batch_to_jsonl(batch_data, output_path):
    """Write a batch of data to a JSONL file (append mode)"""
    with open(output_path, 'a') as f:
        for item in batch_data:
            f.write(json.dumps(item) + '\n')

def jsonl_to_csv(jsonl_path, csv_path):
    # Read JSONL in chunks to avoid loading everything into memory
    with open(jsonl_path, 'r') as f:
        total_lines = sum(1 for _ in f)

    # Process in chunks
    chunksize = 1000
    processed = 0
    first_chunk = True

    with open(jsonl_path, 'r') as f:
        lines_buffer = []

        for i, line in enumerate(tqdm(f, total=total_lines, desc="Converting to CSV")):
            lines_buffer.append(json.loads(line))

            if len(lines_buffer) >= chunksize:
                chunk_df = pd.DataFrame(lines_buffer)

                if first_chunk:
                    chunk_df.to_csv(csv_path, index=False)
                    first_chunk = False
                else:
                    chunk_df.to_csv(csv_path, index=False, mode='a', header=False)

                processed += len(lines_buffer)
                lines_buffer = []

        # Process any remaining lines
        if lines_buffer:
            chunk_df = pd.DataFrame(lines_buffer)
            if first_chunk:
                chunk_df.to_csv(csv_path, index=False)
            else:
                chunk_df.to_csv(csv_path, index=False, mode='a', header=False)

    print(f"Successfully converted {total_lines} records to CSV")


In [41]:
get_train_dataset(seeker_train_path, batch_size=500, use_mentioned_movie_only=True, shuffle_dialgoue=True)

Processing GoRecDial_Train File: 80844it [00:01, 63273.40it/s]
Converting to CSV: 100%|██████████| 8238/8238 [00:00<00:00, 32212.23it/s]

Successfully converted 8238 records to CSV


'GoRecDial_training.csv'

In [9]:
def get_test_dataset(path, name='GoRecDial_test', batch_size=1000):
    # Create output file names
    json_path = f"datasets/GoRecDial/GoRecDial_test.jsonl"
    csv_path = f"datasets/GoRecDial/GoRecDial_test.csv"

    # Clear any existing output files
    with open(json_path, 'w') as f:
        pass

    batch_data = []
    tmp = {'test_inputs': '', 'test_outputs': ''}
    counter = 0

    # Process in batches
    with open(path, 'r') as seeker_train_file:
        for seeker_line in tqdm(seeker_train_file, desc=f"Processing {name} File"):
            if not seeker_line.strip():
                continue

            if 'turn:0' in seeker_line:
                if tmp['test_outputs'] :
                    # Add completed conversation to batch
                    batch_data.append(tmp.copy())
                    counter += 1

                    # Write batch to disk if it reaches batch size
                    if counter % batch_size == 0:
                        write_batch_to_jsonl(batch_data, json_path)
                        batch_data = []

                tmp = {'test_inputs': '', 'test_outputs': ''} # reset tmp
                movie_list = re.findall(MID_pattern, seeker_line)
                tmp['test_outputs'] = seek_test_movieName_dict[movie_list[-1]]

                try:
                    the_first_question = re.findall(r'labels:(.+?)(?=\s*reward:)', seeker_line)
                    if the_first_question and len(the_first_question) > 0:
                        if not any(char in the_first_question[0] for char in ['.', ',', '?', '!']):
                            tmp['test_inputs'] += the_first_question[0] + '.'
                        else:
                            tmp['test_inputs'] += the_first_question[0]
                except Exception as e:
                    print(f"Error processing first question: {e}")
                    print(seeker_line)

            # Process dialogue lines
            else:
                try:
                    text = re.findall(r'text:(.*?)(?=\tlabels:)', seeker_line)
                    response_for_text = re.findall(r'labels:(.+?)(?=\s*reward:)', seeker_line)

                    if text and response_for_text and len(text) > 0 and len(response_for_text) > 0:
                      # Exclude recommend ground truth
                      if movie_list[-1] not in text[0] and movie_list[-1] not in response_for_text[0]:
                        # Replace any MID with real movie name
                        if 'MID' in text[0]:
                          for mid in re.findall(MID_pattern, text[0]):
                            text[0] = text[0].replace(mid, seek_test_movieName_dict[mid])

                        if 'MID' in response_for_text[0]:
                          for mid in re.findall(MID_pattern, response_for_text[0]):
                            response_for_text[0] = response_for_text[0].replace(mid, seek_test_movieName_dict[mid])

                        if not any(char in response_for_text[0] for char in ['.', ',', '?', '!']):
                            tmp['test_inputs'] += text[0] + response_for_text[0] + '.'
                        else:
                            tmp['test_inputs'] += text[0] + response_for_text[0]
                except Exception as e:
                    print(seeker_line)
                    continue

    # Add the last conversation and write remaining batch
    if len(tmp['test_outputs']) != 0:
        batch_data.append(tmp.copy())

    if batch_data:
        write_batch_to_jsonl(batch_data, json_path)

    # Convert the JSONL file to CSV
    jsonl_to_csv(json_path, csv_path)

    return csv_path

def write_batch_to_jsonl(batch_data, output_path):
    """Write a batch of data to a JSONL file (append mode)"""
    with open(output_path, 'a') as f:
        for item in batch_data:
            f.write(json.dumps(item) + '\n')

def jsonl_to_csv(jsonl_path, csv_path):
    # Read JSONL in chunks to avoid loading everything into memory
    with open(jsonl_path, 'r') as f:
        # Count total lines for progress bar
        total_lines = sum(1 for _ in f)

    # Process in chunks
    chunksize = 1000
    processed = 0
    first_chunk = True

    with open(jsonl_path, 'r') as f:
        lines_buffer = []

        for i, line in enumerate(tqdm(f, total=total_lines, desc="Converting to CSV")):
            lines_buffer.append(json.loads(line))

            if len(lines_buffer) >= chunksize:
                chunk_df = pd.DataFrame(lines_buffer)

                if first_chunk:
                    chunk_df.to_csv(csv_path, index=False)
                    first_chunk = False
                else:
                    chunk_df.to_csv(csv_path, index=False, mode='a', header=False)

                processed += len(lines_buffer)
                lines_buffer = []  # Clear buffer

        # Process any remaining lines
        if lines_buffer:
            chunk_df = pd.DataFrame(lines_buffer)
            if first_chunk:
                chunk_df.to_csv(csv_path, index=False)
            else:
                chunk_df.to_csv(csv_path, index=False, mode='a', header=False)

    print(f"Successfully converted {total_lines} records to CSV")


In [10]:
get_test_dataset(seeker_test_path, batch_size=500)

Processing GoRecDial_test File: 8800it [00:00, 68111.84it/s]
Converting to CSV: 100%|██████████| 916/916 [00:00<00:00, 175681.67it/s]

Successfully converted 916 records to CSV


'GoRecDial_test.csv'

In [ ]:
# mid_to_moiveName = {} # If needed

In [ ]:
# # Parse the name of recommend grouth truth movie
# with open(expert_train_path, 'r') as expert_train_file:
#   for expert_line in tqdm(expert_train_file, desc="Processing Expert File"):
#     if 'turn:0' in expert_line:
#       # Parse recommend ground truth movie id and name. e.g. [('MID4226')]

#       try:
#         recommend_ground_truth_movie_id = re.findall(recommend_ground_truth, expert_line)
#         pattern = r"" + recommend_ground_truth_movie_id[0] + r":(.+?) is a"
#         ground_truth_movieName = re.findall(pattern, expert_line)
#         mid_to_moiveName[recommend_ground_truth_movie_id[0]] = ground_truth_movieName[0]
#       except:
#         print(recommend_ground_truth_movie_id)
#         print(ground_truth_movieName)
#         print(expert_line)
#         break

In [ ]:
# # Parse the data
# count = 0

# This method run TOO SLOW!!!
# def get_dataset (path):
#   tmp = {'movies':[], 'full_situation':''} # used to construct final csv
#   final_csv = []
#   with open(path, 'r') as seeker_train_file:
#     for seeker_line in tqdm(seeker_train_file, desc="Processing Seeker File"):

#       # Not EOF
#       if seeker_line:
#         if 'turn:0' in seeker_line:
#           if len(tmp['movies']) != 0:
#             final_csv.append(tmp.copy())
#           else:
#             tmp = {'movies': [], 'full_situation': ''} # reset it for the new line

#           movie_list = re.findall(MID_pattern, seeker_line)

#           # Update tmp
#           tmp['movies'] = movie_list
#           the_first_question = re.findall(r'labels:(.+?)(?=\s*reward:)', seeker_line)
#           if not any(char in the_first_question for char in ['.', ',', '?', '!']):
#             tmp['full_situation'] += the_first_question[0] + '.'
#           else:
#             tmp['full_situation'] += the_first_question[0]

#         # Dialogue line
#         else:
#           text = re.findall(r'text:(.*?)(?=\tlabels:)', seeker_line)
#           response_for_text = re.findall( r'labels:(.+?)(?=\s*reward:)',seeker_line)
#           try:
#             if not any(char in response_for_text for char in ['.', ',', '?', '!']):
#               tmp['full_situation'] += text[0] + response_for_text[0] + '.'
#             else:
#               tmp['full_situation'] += text[0] + response_for_text[0]
#           except:
#             print(text)
#             print(response_for_text)
#             print(seeker_line)
#             break

#       # EOF
#       else:
#         final_csv.append(tmp.copy())

#     dataset = pd.DataFrame(final_csv)

#   return dataset



In [ ]:
dict1 = {'a': 1, 'b': 2}
dict2 = {'a': 3, 'd': 4}
dict3 = {**dict1, **dict2}
print(dict3)
#

{'a': 3, 'b': 2, 'd': 4}
